# 03 — Baseline Models
**EduForecast** | Logistic Regression & Random Forest | Notebook 3 of 6

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve,
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import PROCESSED_DIR, FIGURES_DIR, METRICS_DIR, RANDOM_SEED

np.random.seed(RANDOM_SEED)
sns.set_style('whitegrid')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
DPI = 150

print('Environment ready.')

## 1. Load Processed Data

In [ ]:
print('Loading processed static features...')

train_df = pd.read_csv(PROCESSED_DIR / 'train_static.csv')
test_df  = pd.read_csv(PROCESSED_DIR / 'test_static.csv')

print(f'train_static : {train_df.shape}')
print(f'test_static  : {test_df.shape}')

# Drop non-feature columns — keep only numeric model inputs
DROP_COLS = ['id_student', 'code_module', 'code_presentation', 'is_at_risk']
FEATURE_COLS = [c for c in train_df.columns if c not in DROP_COLS]

X_train = train_df[FEATURE_COLS].values
y_train = train_df['is_at_risk'].values

X_test  = test_df[FEATURE_COLS].values
y_test  = test_df['is_at_risk'].values

print(f'\nX_train shape : {X_train.shape}')
print(f'X_test shape  : {X_test.shape}')
print(f'Features      : {len(FEATURE_COLS)}')
print(f'\nClass distribution in train:')
unique, counts = np.unique(y_train, return_counts=True)
for cls, cnt in zip(unique, counts):
    label = 'At-Risk' if cls == 1 else 'Not At-Risk'
    print(f'  {label} ({cls}): {cnt:,}  ({cnt/len(y_train)*100:.1f}%)')

## 2. Preprocessing

In [ ]:
print('Fitting StandardScaler on training data...')

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

assert not np.isnan(X_train_sc).any(), 'NaN in scaled X_train'
assert not np.isnan(X_test_sc).any(),  'NaN in scaled X_test'

print(f'X_train mean after scaling : {X_train_sc.mean():.6f}  (≈ 0)')
print(f'X_train std  after scaling : {X_train_sc.std():.6f}   (≈ 1)')
print('Preprocessing complete.')

## 3. Logistic Regression

In [ ]:
print('Training Logistic Regression...')

lr = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_SEED,
    class_weight='balanced',
)
lr.fit(X_train_sc, y_train)
print('Training complete.')

lr_pred  = lr.predict(X_test_sc)
lr_prob  = lr.predict_proba(X_test_sc)[:, 1]

lr_acc  = accuracy_score(y_test, lr_pred)
lr_prec = precision_score(y_test, lr_pred, average='weighted', zero_division=0)
lr_rec  = recall_score(y_test, lr_pred, average='weighted', zero_division=0)
lr_f1   = f1_score(y_test, lr_pred, average='weighted', zero_division=0)
lr_auc  = roc_auc_score(y_test, lr_prob)

print()
print('LOGISTIC REGRESSION RESULTS')
print(f'  Accuracy  : {lr_acc:.4f}')
print(f'  Precision : {lr_prec:.4f}')
print(f'  Recall    : {lr_rec:.4f}')
print(f'  F1 Score  : {lr_f1:.4f}')
print(f'  AUC-ROC   : {lr_auc:.4f}')

print()
print('Classification Report:')
print(classification_report(y_test, lr_pred,
                            target_names=['Not At-Risk', 'At-Risk']))

In [ ]:
lr_metrics = {
    'model':     'logistic_regression',
    'accuracy':  round(lr_acc,  4),
    'precision': round(lr_prec, 4),
    'recall':    round(lr_rec,  4),
    'f1':        round(lr_f1,   4),
    'auc_roc':   round(lr_auc,  4),
}
pd.DataFrame([lr_metrics]).to_csv(
    METRICS_DIR / 'logistic_regression_metrics.csv', index=False
)
print('Saved → logistic_regression_metrics.csv')

## 4. Random Forest

In [ ]:
print('Training Random Forest (n_estimators=200)...')

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=RANDOM_SEED,
    class_weight='balanced',
    n_jobs=-1,
)
rf.fit(X_train_sc, y_train)
print('Training complete.')

rf_pred  = rf.predict(X_test_sc)
rf_prob  = rf.predict_proba(X_test_sc)[:, 1]

rf_acc  = accuracy_score(y_test, rf_pred)
rf_prec = precision_score(y_test, rf_pred, average='weighted', zero_division=0)
rf_rec  = recall_score(y_test, rf_pred, average='weighted', zero_division=0)
rf_f1   = f1_score(y_test, rf_pred, average='weighted', zero_division=0)
rf_auc  = roc_auc_score(y_test, rf_prob)

print()
print('RANDOM FOREST RESULTS')
print(f'  Accuracy  : {rf_acc:.4f}')
print(f'  Precision : {rf_prec:.4f}')
print(f'  Recall    : {rf_rec:.4f}')
print(f'  F1 Score  : {rf_f1:.4f}')
print(f'  AUC-ROC   : {rf_auc:.4f}')

print()
print('Classification Report:')
print(classification_report(y_test, rf_pred,
                            target_names=['Not At-Risk', 'At-Risk']))

In [ ]:
rf_metrics = {
    'model':     'random_forest',
    'accuracy':  round(rf_acc,  4),
    'precision': round(rf_prec, 4),
    'recall':    round(rf_rec,  4),
    'f1':        round(rf_f1,   4),
    'auc_roc':   round(rf_auc,  4),
}
pd.DataFrame([rf_metrics]).to_csv(
    METRICS_DIR / 'random_forest_metrics.csv', index=False
)
print('Saved → random_forest_metrics.csv')

## 5. Results Comparison

In [ ]:
comparison = pd.DataFrame([lr_metrics, rf_metrics])
comparison['model'] = ['Logistic Regression', 'Random Forest']

print('Model comparison:')
print(comparison.to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
# ── Plot 1: Grouped bar chart — metric comparison ──────────────────────────
metric_names = ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']
display_names = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC']
models = ['Logistic Regression', 'Random Forest']
model_colors = ['#3498db', '#e74c3c']

lr_vals = [lr_metrics[m] for m in metric_names]
rf_vals = [rf_metrics[m] for m in metric_names]

x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
bars_lr = ax.bar(x - width/2, lr_vals, width, label='Logistic Regression',
                 color=model_colors[0], alpha=0.85, edgecolor='white')
bars_rf = ax.bar(x + width/2, rf_vals, width, label='Random Forest',
                 color=model_colors[1], alpha=0.85, edgecolor='white')

for bars in (bars_lr, bars_rf):
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                f'{h:.3f}', ha='center', va='bottom', fontsize=8.5)

ax.set_xticks(x)
ax.set_xticklabels(display_names, fontsize=11)
ax.set_ylabel('Score', fontsize=12)
ax.set_ylim(0, 1.12)
ax.set_title('Baseline Models — Performance Comparison', fontsize=14, pad=12)
ax.legend(fontsize=11)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
plt.tight_layout()
out1 = FIGURES_DIR / '03_baseline_comparison.png'
fig.savefig(out1, dpi=DPI)
plt.close(fig)
print(f'Saved \u2192 {out1}')

In [ ]:
# ── Plot 2: ROC curves ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.500)')

for label, prob, auc_val, color in [
    ('Logistic Regression', lr_prob, lr_auc, model_colors[0]),
    ('Random Forest',       rf_prob, rf_auc, model_colors[1]),
]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax.plot(fpr, tpr, color=color, linewidth=2.2,
            label=f'{label}  (AUC = {auc_val:.4f})')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves \u2014 Baseline Models', fontsize=14, pad=12)
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
plt.tight_layout()
out2 = FIGURES_DIR / '03_roc_curves_baseline.png'
fig.savefig(out2, dpi=DPI)
plt.close(fig)
print(f'Saved \u2192 {out2}')

## 6. Feature Importance

In [ ]:
# ── Random Forest feature importances ─────────────────────────────────────
importances = rf.feature_importances_
feat_imp = (
    pd.DataFrame({'feature': FEATURE_COLS, 'importance': importances})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)

print('Top 10 features by Random Forest importance:')
print(f'{"Rank":<5} {"Feature":<32} {"Importance":>12}')
print('-' * 50)
for i, row in feat_imp.head(10).iterrows():
    print(f'{i+1:<5} {row["feature"]:<32} {row["importance"]:>12.6f}')

In [ ]:
# ── Plot 3: Top-15 feature importance bar chart ────────────────────────────
top15 = feat_imp.head(15).iloc[::-1]  # reverse for horizontal bar (highest at top)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top15['feature'], top15['importance'],
               color='#3498db', edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, top15['importance']):
    ax.text(val + 0.0005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

ax.set_xlabel('Feature Importance (Mean Decrease in Impurity)', fontsize=11)
ax.set_title('Random Forest \u2014 Top 15 Feature Importances', fontsize=13, pad=12)
ax.set_xlim(0, top15['importance'].max() * 1.18)
plt.tight_layout()
out3 = FIGURES_DIR / '03_rf_feature_importance.png'
fig.savefig(out3, dpi=DPI)
plt.close(fig)
print(f'Saved \u2192 {out3}')

In [ ]:
# ── Confusion matrix statistics ────────────────────────────────────────────
cm = confusion_matrix(y_test, rf_pred)
tn, fp, fn, tp = cm.ravel()

print('Random Forest — Confusion Matrix')
print(f'  True Negatives  (TN): {tn:,}  (correctly identified Not At-Risk)')
print(f'  False Positives (FP): {fp:,}  (Not At-Risk predicted as At-Risk)')
print(f'  False Negatives (FN): {fn:,}  (At-Risk missed by model)')
print(f'  True Positives  (TP): {tp:,}  (correctly identified At-Risk)')
print()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
print(f'  Sensitivity (Recall for At-Risk): {sensitivity:.4f}')
print(f'  Specificity (Recall for Not At-Risk): {specificity:.4f}')

In [ ]:
# ── Plot 4: Confusion matrix heatmap ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Not At-Risk', 'At-Risk'],
)
disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format=',')
ax.set_title('Random Forest \u2014 Confusion Matrix', fontsize=13, pad=12)

# Annotate with percentages
total = cm.sum()
for i in range(2):
    for j in range(2):
        pct = cm[i, j] / total * 100
        ax.text(j, i + 0.35, f'({pct:.1f}%)',
                ha='center', va='center', fontsize=10,
                color='white' if i == j else 'black')

plt.tight_layout()
out4 = FIGURES_DIR / '03_rf_confusion_matrix.png'
fig.savefig(out4, dpi=DPI)
plt.close(fig)
print(f'Saved \u2192 {out4}')

## 7. Save All Metrics

In [ ]:
# Save combined comparison CSV
comparison_out = comparison.copy()
comparison_out.to_csv(METRICS_DIR / 'baseline_comparison.csv', index=False)
print('Saved \u2192 baseline_comparison.csv')

print('\nAll figures saved:')
for f in sorted(FIGURES_DIR.glob('03_*.png')):
    print(f'  {f.name}')

In [ ]:
best_model = 'Random Forest' if rf_auc >= lr_auc else 'Logistic Regression'
best_auc   = max(lr_auc, rf_auc)

print('=' * 55)
print('BASELINE MODELS SUMMARY')
print('=' * 55)
print(f'{"Model":<25} {"AUC":>6} {"F1":>6} {"Acc":>6}')
print('-' * 55)
print(f'{"Logistic Regression":<25} {lr_auc:>6.4f} {lr_f1:>6.4f} {lr_acc:>6.4f}')
print(f'{"Random Forest":<25} {rf_auc:>6.4f} {rf_f1:>6.4f} {rf_acc:>6.4f}')
print('=' * 55)
print(f'Best baseline model: {best_model}')
print(f'Best AUC: {best_auc:.4f}')
print('=' * 55)